## Example 1: Basic Decorator Creation

In [ ]:
import decorator
import time
import functools

print("Example 1: Basic Decorator Creation")

try:
    print("✓ decorator package information:")
    print(f"  Module type: {type(decorator).__name__}")
    
    # Show key components
    attrs = [attr for attr in dir(decorator) if not attr.startswith('_')]
    print(f"  Public attributes: {len(attrs)}")
    key_attrs = [attr for attr in attrs if attr in ['decorator', 'decorate', 'dispatch_on']]
    print(f"  Key functions: {key_attrs}")
    
    # Basic decorator using decorator.decorator
    print("✓ Basic decorator creation:")
    
    def timing_decorator(func, *args, **kwargs):
        """Decorator that measures function execution time"""
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        print(f"  Function '{func.__name__}' took {end_time - start_time:.4f} seconds")
        return result
    
    # Create decorator using decorator.decorator
    timing = decorator.decorator(timing_decorator)
    
    # Test function with our decorator
    @timing
    def slow_function(n):
        """A function that takes some time to execute"""
        total = 0
        for i in range(n):
            total += i * i
        return total
    
    print(f"  Testing timing decorator:")
    result = slow_function(1000)
    print(f"  Result: {result}")
    
    # Show preserved function metadata
    print("✓ Function metadata preservation:")
    
    print(f"  Function name: {slow_function.__name__}")
    print(f"  Function docstring: {slow_function.__doc__}")
    print(f"  Function module: {slow_function.__module__}")
    
    # Compare with traditional decorator (loses metadata)
    print("✓ Traditional decorator comparison:")
    
    def traditional_timing(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            start_time = time.time()
            result = func(*args, **kwargs)
            end_time = time.time()
            print(f"  Traditional: '{func.__name__}' took {end_time - start_time:.4f} seconds")
            return result
        return wrapper
    
    @traditional_timing
    def another_slow_function(n):
        """Another slow function"""
        return sum(i**2 for i in range(n))
    
    print(f"  Traditional decorator result:")
    trad_result = another_slow_function(1000)
    print(f"  Result: {trad_result}")
    print(f"  Name preserved: {another_slow_function.__name__}")
    
    # Demonstrate signature preservation
    print("✓ Signature preservation:")
    
    def log_calls_decorator(func, *args, **kwargs):
        """Decorator that logs function calls"""
        print(f"  Calling {func.__name__} with args={args}, kwargs={kwargs}")
        return func(*args, **kwargs)
    
    log_calls = decorator.decorator(log_calls_decorator)
    
    @log_calls
    def complex_function(a, b, c=10, *args, **kwargs):
        """A function with complex signature"""
        return a + b + c + sum(args) + sum(kwargs.values())
    
    # Test signature preservation
    print(f"  Testing signature preservation:")
    result = complex_function(1, 2, 3, 4, 5, x=6, y=7)
    print(f"  Complex function result: {result}")
    
    # Show that signature is preserved for introspection
    import inspect
    sig = inspect.signature(complex_function)
    print(f"  Function signature: {sig}")
    
    print("✓ Basic decorator creation completed successfully")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("  Note: decorator package simplifies decorator creation")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"decorator example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 1 finished")

## Example 2: Advanced Decorator Patterns

In [ ]:
import decorator

print("Example 2: Advanced Decorator Patterns")

try:
    # Parameterized decorators
    print("✓ Parameterized decorators:")
    
    def retry_decorator(max_attempts=3):
        """Create a retry decorator with configurable attempts"""
        def actual_decorator(func, *args, **kwargs):
            last_exception = None
            for attempt in range(max_attempts):
                try:
                    result = func(*args, **kwargs)
                    if attempt > 0:
                        print(f"  Function '{func.__name__}' succeeded on attempt {attempt + 1}")
                    return result
                except Exception as e:
                    last_exception = e
                    print(f"  Attempt {attempt + 1} failed: {e}")
                    if attempt == max_attempts - 1:
                        print(f"  All {max_attempts} attempts failed")
                        raise last_exception
            return None
        
        return decorator.decorator(actual_decorator)
    
    # Test parameterized decorator
    @retry_decorator(max_attempts=3)
    def unreliable_function(success_rate):
        """Function that randomly succeeds or fails"""
        import random
        if random.random() < success_rate:
            return "Success!"
        else:
            raise ValueError("Random failure")
    
    print(f"  Testing retry decorator:")
    try:
        # Set seed for reproducible behavior
        import random
        random.seed(42)
        result = unreliable_function(0.3)  # 30% success rate
        print(f"  Result: {result}")
    except Exception as e:
        print(f"  Final failure: {e}")
        raise
    
    # Caching decorator
    print("✓ Caching decorator:")
    
    def memoize_decorator(func, *args, **kwargs):
        """Decorator that caches function results"""
        # Create cache attribute if it doesn't exist
        if not hasattr(func, '_cache'):
            func._cache = {}
        
        # Create cache key from arguments
        key = str(args) + str(sorted(kwargs.items()))
        
        if key in func._cache:
            print(f"  Cache hit for {func.__name__}")
            return func._cache[key]
        
        print(f"  Cache miss for {func.__name__}, computing...")
        result = func(*args, **kwargs)
        func._cache[key] = result
        return result
    
    memoize = decorator.decorator(memoize_decorator)
    
    @memoize
    def fibonacci(n):
        """Compute fibonacci number (slow without caching)"""
        if n < 2:
            return n
        return fibonacci(n-1) + fibonacci(n-2)
    
    print(f"  Testing memoization:")
    print(f"  fibonacci(10) = {fibonacci(10)}")
    print(f"  fibonacci(10) = {fibonacci(10)}")  # Should hit cache
    print(f"  fibonacci(15) = {fibonacci(15)}")  # Will use cached values
    
    # Input validation decorator
    print("✓ Input validation decorator:")
    
    def validate_types(**type_checks):
        """Create decorator that validates argument types"""
        def actual_decorator(func, *args, **kwargs):
            import inspect
            
            # Get function signature
            sig = inspect.signature(func)
            bound_args = sig.bind(*args, **kwargs)
            bound_args.apply_defaults()
            
            # Validate types
            for param_name, expected_type in type_checks.items():
                if param_name in bound_args.arguments:
                    value = bound_args.arguments[param_name]
                    if not isinstance(value, expected_type):
                        raise TypeError(
                            f"Parameter '{param_name}' must be {expected_type.__name__}, "
                            f"got {type(value).__name__}"
                        )
            
            return func(*args, **kwargs)
        
        return decorator.decorator(actual_decorator)
    
    @validate_types(name=str, age=int, active=bool)
    def create_user(name, age, active=True):
        """Create user with validated inputs"""
        return {"name": name, "age": age, "active": active}
    
    print(f"  Testing input validation:")
    
    # Valid call
    try:
        user1 = create_user("Alice", 25)
        print(f"  Valid user created: {user1}")
    except Exception as e:
        print(f"  Validation error: {e}")
        raise
    
    # Invalid call
    try:
        user2 = create_user("Bob", "twenty-five")  # age should be int
        print(f"  Invalid user created: {user2}")
    except TypeError as e:
        print(f"  ✓ Validation caught error: {e}")
    
    print("✓ Advanced decorator patterns completed successfully")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("  Note: decorator package enables advanced decorator patterns")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"decorator example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: Method Decorators and Class Integration

In [ ]:
import decorator

print("Example 3: Method Decorators and Class Integration")

try:
    # Method decorators
    print("✓ Method decorators:")
    
    def access_control_decorator(required_role):
        """Create decorator for method access control"""
        def actual_decorator(method, self, *args, **kwargs):
            if not hasattr(self, 'role'):
                raise AttributeError("Object must have a 'role' attribute")
            
            if self.role != required_role:
                raise PermissionError(f"Access denied. Required role: {required_role}, current role: {self.role}")
            
            print(f"  Access granted for {method.__name__} (role: {self.role})")
            return method(self, *args, **kwargs)
        
        return decorator.decorator(actual_decorator)
    
    # Test class with decorated methods
    class SecureService:
        def __init__(self, role):
            self.role = role
            self.data = []
        
        @access_control_decorator("admin")
        def admin_operation(self):
            """Operation requiring admin access"""
            self.data.append("admin_data")
            return "Admin operation completed"
        
        @access_control_decorator("user") 
        def user_operation(self):
            """Operation requiring user access"""
            self.data.append("user_data")
            return "User operation completed"
        
        def public_operation(self):
            """Public operation (no decorator)"""
            return f"Public data: {len(self.data)} items"
    
    # Test access control
    print(f"  Testing access control:")
    
    admin_service = SecureService("admin")
    user_service = SecureService("user")
    
    # Admin can do admin operations
    try:
        result = admin_service.admin_operation()
        print(f"    Admin->admin: {result}")
    except Exception as e:
        print(f"    Admin->admin failed: {e}")
        raise
    
    # User cannot do admin operations
    try:
        result = user_service.admin_operation()
        print(f"    User->admin: {result}")
    except PermissionError as e:
        print(f"    ✓ User->admin blocked: {e}")
    
    # User can do user operations
    try:
        result = user_service.user_operation()
        print(f"    User->user: {result}")
    except Exception as e:
        print(f"    User->user failed: {e}")
        raise
    
    # Property decorators
    print("✓ Property decorators:")
    
    def logged_property_decorator(func, self, *args, **kwargs):
        """Decorator that logs property access"""
        prop_name = func.__name__.replace('get_', '').replace('set_', '')
        if args:  # setter
            print(f"  Setting property '{prop_name}' to {args[0]}")
        else:  # getter
            print(f"  Getting property '{prop_name}'")
        return func(self, *args, **kwargs)
    
    logged_property = decorator.decorator(logged_property_decorator)
    
    class LoggedClass:
        def __init__(self):
            self._value = 0
            self._name = "default"
        
        @logged_property
        def get_value(self):
            return self._value
        
        @logged_property  
        def set_value(self, val):
            self._value = val
        
        value = property(get_value, set_value)
        
        @logged_property
        def get_name(self):
            return self._name
        
        @logged_property
        def set_name(self, name):
            self._name = name
            
        name = property(get_name, set_name)
    
    print(f"  Testing logged properties:")
    obj = LoggedClass()
    obj.value = 42
    retrieved_value = obj.value
    print(f"    Retrieved value: {retrieved_value}")
    
    obj.name = "test_object"
    retrieved_name = obj.name
    print(f"    Retrieved name: {retrieved_name}")
    
    # Class method decorators
    print("✓ Class and static method decorators:")
    
    def benchmark_decorator(func, *args, **kwargs):
        """Decorator that benchmarks method calls"""
        import time
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        
        # Determine method type
        if args and hasattr(args[0], '__class__'):
            method_type = "instance method"
        elif hasattr(func, '__self__') and func.__self__ is not None:
            method_type = "class method"
        else:
            method_type = "static method"
            
        print(f"  {method_type} '{func.__name__}' took {end_time - start_time:.6f}s")
        return result
    
    benchmark = decorator.decorator(benchmark_decorator)
    
    class BenchmarkedClass:
        counter = 0
        
        @benchmark
        def instance_method(self, n):
            """Benchmarked instance method"""
            return sum(range(n))
        
        @classmethod
        @benchmark
        def class_method(cls, n):
            """Benchmarked class method"""
            cls.counter += 1
            return cls.counter * n
        
        @staticmethod
        @benchmark
        def static_method(n):
            """Benchmarked static method"""
            return n ** 2
    
    print(f"  Testing method type detection:")
    obj = BenchmarkedClass()
    
    instance_result = obj.instance_method(1000)
    print(f"    Instance method result: {instance_result}")
    
    class_result = BenchmarkedClass.class_method(5)
    print(f"    Class method result: {class_result}")
    
    static_result = BenchmarkedClass.static_method(10)
    print(f"    Static method result: {static_result}")
    
    print("✓ Method decorators and class integration completed successfully")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("  Note: decorator package works seamlessly with class methods")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"decorator example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: Error Handling and Debugging

In [ ]:
import decorator
import traceback
import sys

print("Example 4: Error Handling and Debugging")

try:
    # Error handling decorator
    print("✓ Error handling decorator:")
    
    def error_handler_decorator(func, *args, **kwargs):
        """Decorator that handles exceptions gracefully"""
        try:
            return func(*args, **kwargs)
        except Exception as e:
            error_info = {
                'function': func.__name__,
                'args': args,
                'kwargs': kwargs,
                'error_type': type(e).__name__,
                'error_message': str(e)
            }
            print(f"  Error in {func.__name__}: {error_info['error_type']}")
            print(f"  Message: {error_info['error_message']}")
            print(f"  Args: {args}, Kwargs: {kwargs}")
            
            # Return None or raise based on error type
            if isinstance(e, (ValueError, TypeError, ZeroDivisionError)):
                print(f"  Returning None due to handled error")
                return None
            else:
                print(f"  Re-raising unexpected error")
                raise
    
    error_handler = decorator.decorator(error_handler_decorator)
    
    @error_handler
    def divide_numbers(a, b):
        """Function that might divide by zero"""
        if b == 0:
            raise ZeroDivisionError("Cannot divide by zero")
        return a / b
    
    @error_handler
    def parse_number(text):
        """Function that might fail to parse"""
        return int(text)
    
    print(f"  Testing error handling:")
    
    # Successful operations
    result1 = divide_numbers(10, 2)
    print(f"    10 / 2 = {result1}")
    
    result2 = parse_number("42")
    print(f"    parse('42') = {result2}")
    
    # Operations that cause handled errors
    result3 = divide_numbers(10, 0)  # ZeroDivisionError (now handled -> None)
    print(f"    10 / 0 = {result3}")
    
    result4 = parse_number("not_a_number")  # ValueError (handled -> None)
    print(f"    parse('not_a_number') = {result4}")
    
    # Debugging decorator
    print("✓ Debugging decorator:")
    
    def debug_decorator(func, *args, **kwargs):
        """Decorator that provides detailed debugging info"""
        import inspect
        
        # Get function signature
        sig = inspect.signature(func)
        bound_args = sig.bind(*args, **kwargs)
        bound_args.apply_defaults()
        
        print(f"  Debugging {func.__name__}:")
        print(f"    Module: {func.__module__}")
        print(f"    Signature: {sig}")
        print(f"    Bound args: {dict(bound_args.arguments)}")
        
        # Execute function with timing
        import time
        start_time = time.time()
        
        try:
            result = func(*args, **kwargs)
            end_time = time.time()
            
            print(f"    Execution time: {end_time - start_time:.6f}s")
            print(f"    Return value: {result} ({type(result).__name__})")
            return result
            
        except Exception as e:
            end_time = time.time()
            print(f"    Execution time: {end_time - start_time:.6f}s")
            print(f"    Exception: {type(e).__name__}: {e}")
            # Don't print full traceback in example to keep output clean
            print(f"    Exception handled by debug decorator")
            raise
    
    debug = decorator.decorator(debug_decorator)
    
    @debug
    def complex_calculation(x, y, operation="add"):
        """Function with complex logic for debugging"""
        operations = {
            "add": lambda a, b: a + b,
            "multiply": lambda a, b: a * b,
            "power": lambda a, b: a ** b,
            "divide": lambda a, b: a / b if b != 0 else None
        }
        
        if operation not in operations:
            raise ValueError(f"Unknown operation: {operation}")
        
        return operations[operation](x, y)
    
    print(f"  Testing debugging decorator:")
    
    result = complex_calculation(5, 3, "multiply")
    print(f"    Result received: {result}")
    
    # Test with safe division
    result_div = complex_calculation(10, 2, "divide")
    print(f"    Division result: {result_div}")
    
    # Performance monitoring decorator
    print("✓ Performance monitoring decorator:")
    
    def performance_monitor_decorator(func, *args, **kwargs):
        """Decorator that monitors function performance"""
        import time
        
        # Before execution
        start_time = time.time()
        
        try:
            result = func(*args, **kwargs)
            
            # After execution
            end_time = time.time()
            
            # Calculate metrics
            execution_time = end_time - start_time
            
            print(f"  Performance metrics for {func.__name__}:")
            print(f"    Execution time: {execution_time:.6f}s")
            print(f"    Function completed successfully")
            
            return result
            
        except Exception as e:
            end_time = time.time()
            execution_time = end_time - start_time
            print(f"  Performance metrics for {func.__name__} (failed):")
            print(f"    Execution time before error: {execution_time:.6f}s")
            raise
    
    performance_monitor = decorator.decorator(performance_monitor_decorator)
    
    @performance_monitor
    def memory_intensive_function(size):
        """Function that uses memory"""
        data = list(range(size))
        return sum(data)
    
    print(f"  Testing performance monitoring:")
    result = memory_intensive_function(10000)
    print(f"    Result: {result}")
    
    # Stack trace decorator
    print("✓ Stack trace decorator:")
    
    def stack_trace_decorator(func, *args, **kwargs):
        """Decorator that shows call stack information"""
        import inspect
        
        # Get current stack
        stack = inspect.stack()
        
        print(f"  Call stack for {func.__name__}:")
        # Skip the decorator frames, show calling context
        for i, frame_info in enumerate(stack[2:4]):  # Show 2 frames to keep output manageable
            filename = frame_info.filename.split('/')[-1]  # Just filename
            print(f"    {i+1}. {filename}:{frame_info.lineno} in {frame_info.function}")
        
        return func(*args, **kwargs)
    
    stack_trace = decorator.decorator(stack_trace_decorator)
    
    @stack_trace
    def traced_function():
        """Function that shows its call stack"""
        return "Function executed with stack trace"
    
    def caller_function():
        return traced_function()
    
    def wrapper_function():
        return caller_function()
    
    print(f"  Testing stack trace decorator:")
    result = wrapper_function()
    print(f"    Result: {result}")
    
    # Safe error propagation example
    print("✓ Safe error propagation:")
    
    def safe_error_decorator(func, *args, **kwargs):
        """Decorator that safely handles errors without crashing"""
        try:
            return func(*args, **kwargs)
        except Exception as e:
            error_msg = f"Safe handling of {type(e).__name__} in {func.__name__}: {str(e)}"
            print(f"  {error_msg}")
            return {"error": True, "message": str(e), "function": func.__name__}
    
    safe_error = decorator.decorator(safe_error_decorator)
    
    @safe_error
    def risky_operation(x, y):
        """Function that might have various errors"""
        if y == 0:
            return x / y  # This will cause ZeroDivisionError
        elif x < 0:
            raise ValueError("x must be non-negative")
        else:
            return x ** y
    
    print(f"  Testing safe error propagation:")
    
    # Test successful operation
    result1 = risky_operation(2, 3)
    print(f"    risky_operation(2, 3) = {result1}")
    
    # Test division by zero
    result2 = risky_operation(5, 0)
    print(f"    risky_operation(5, 0) = {result2}")
    
    # Test negative input
    result3 = risky_operation(-1, 2)
    print(f"    risky_operation(-1, 2) = {result3}")
    
    print("✓ Error handling and debugging completed successfully")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("  Note: Some debugging features require additional packages")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"decorator example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")

## Example 5: Real-world Applications and Best Practices

In [ ]:
import decorator
import functools
from datetime import datetime

print("Example 5: Real-world Applications and Best Practices")

try:
    # Web API decorators
    print("✓ Web API decorators:")
    
    class RateLimitExceeded(Exception):
        """Raised by the rate-limit decorator below once `max_calls` is reached inside the window."""

    def rate_limit_decorator(max_calls=10, time_window=60):
        """Create a rate limiting decorator"""
        call_history = {}
        
        def actual_decorator(func, *args, **kwargs):
            import time
            current_time = time.time()
            
            # Get caller identifier (simplified - would use IP/user ID in real app)
            caller_id = "default_user"
            if caller_id not in call_history:
                call_history[caller_id] = []
            
            # Clean old calls outside time window
            call_history[caller_id] = [
                call_time for call_time in call_history[caller_id]
                if current_time - call_time < time_window
            ]
            
            # Check rate limit
            if len(call_history[caller_id]) >= max_calls:
                raise RateLimitExceeded(f"Rate limit exceeded: {max_calls} calls per {time_window}s")
            
            # Record this call
            call_history[caller_id].append(current_time)
            
            print(f"  API call to {func.__name__} ({len(call_history[caller_id])}/{max_calls})")
            return func(*args, **kwargs)
        
        return decorator.decorator(actual_decorator)
    
    @rate_limit_decorator(max_calls=3, time_window=5)
    def api_endpoint(user_id):
        """Simulated API endpoint"""
        return {"user_id": user_id, "timestamp": datetime.now().isoformat()}
    
    print(f"  Testing rate limiting:")
    outcomes = []
    for i in range(5):
        try:
            result = api_endpoint(f"user_{i}")
            assert result["user_id"] == f"user_{i}", result
            print(f"    Call {i+1}: Success")
            outcomes.append("ok")
        except RateLimitExceeded as e:
            print(f"    Call {i+1}: {e}")
            outcomes.append("limited")
    # 3 calls per 5 s window, made back to back: the first three pass, the 4th and 5th are refused.
    assert outcomes == ["ok", "ok", "ok", "limited", "limited"], outcomes
    
    # Authentication decorator
    print("✓ Authentication decorator:")
    
    # Simple token store (in real app, would be database/cache)
    valid_tokens = {
        "token123": {"user": "alice", "role": "admin"},
        "token456": {"user": "bob", "role": "user"}
    }
    
    def require_auth_decorator(required_role=None):
        """Create authentication decorator"""
        def actual_decorator(func, *args, **kwargs):
            # In real app, would get token from request headers
            # For demo, assume first arg is token
            if not args or not isinstance(args[0], str) or not args[0].startswith("token"):
                raise ValueError("Missing or invalid authentication token")
            
            token = args[0]
            if token not in valid_tokens:
                raise ValueError("Invalid authentication token")
            
            user_info = valid_tokens[token]
            if required_role and user_info["role"] != required_role:
                raise PermissionError(f"Required role: {required_role}, user role: {user_info['role']}")
            
            print(f"  Authenticated user: {user_info['user']} ({user_info['role']})")
            # Pass user info to function (remove token from args)
            return func(user_info, *args[1:], **kwargs)
        
        return decorator.decorator(actual_decorator)
    
    @require_auth_decorator(required_role="admin")
    def admin_operation(user_info, data):
        """Admin-only operation"""
        return f"Admin {user_info['user']} processed: {data}"
    
    @require_auth_decorator()
    def user_operation(user_info, data):
        """Any authenticated user operation"""
        return f"User {user_info['user']} accessed: {data}"
    
    print(f"  Testing authentication:")
    
    # Valid admin token
    try:
        result = admin_operation("token123", "admin_data")
        print(f"    Admin operation: {result}")
    except Exception as e:
        print(f"    Admin operation failed: {e}")
        raise
    
    # User token trying admin operation
    try:
        result = admin_operation("token456", "admin_data")
    except PermissionError as e:
        assert str(e) == "Required role: admin, user role: user", e
        print(f"    ✓ User->admin blocked: {e}")
    else:
        raise AssertionError(f"a 'user' token passed the admin-only check: {result}")
    
    # Valid user operation
    try:
        result = user_operation("token456", "user_data")
        print(f"    User operation: {result}")
    except Exception as e:
        print(f"    User operation failed: {e}")
        raise
    
    # Caching with expiration
    print("✓ Advanced caching decorator:")
    
    def cache_with_ttl_decorator(ttl_seconds=30):
        """Create caching decorator with time-to-live"""
        cache = {}
        
        def actual_decorator(func, *args, **kwargs):
            import time
            current_time = time.time()
            
            # Create cache key
            cache_key = f"{func.__name__}:{str(args)}:{str(sorted(kwargs.items()))}"
            
            # Check if cached and not expired
            if cache_key in cache:
                cached_time, cached_result = cache[cache_key]
                if current_time - cached_time < ttl_seconds:
                    print(f"  Cache hit for {func.__name__} (age: {current_time - cached_time:.1f}s)")
                    return cached_result
                else:
                    print(f"  Cache expired for {func.__name__} (age: {current_time - cached_time:.1f}s)")
                    del cache[cache_key]
            
            # Execute function and cache result
            print(f"  Cache miss for {func.__name__}, executing...")
            result = func(*args, **kwargs)
            cache[cache_key] = (current_time, result)
            print(f"  Cached result for {cache_key}")
            
            return result
        
        return decorator.decorator(actual_decorator)
    
    @cache_with_ttl_decorator(ttl_seconds=2)
    def expensive_calculation(n):
        """Simulate expensive calculation"""
        import time
        time.sleep(0.1)  # Simulate work
        return sum(i**2 for i in range(n))
    
    print(f"  Testing TTL caching:")
    
    # First call - cache miss
    result1 = expensive_calculation(100)
    print(f"    Result 1: {result1}")
    
    # Second call - cache hit
    result2 = expensive_calculation(100)
    print(f"    Result 2: {result2}")
    
    # Wait for cache to expire and call again
    import time
    time.sleep(2.1)
    result3 = expensive_calculation(100)
    print(f"    Result 3 (after expiry): {result3}")
    
    # Best practices demonstration
    print("✓ Best practices:")
    
    best_practices = [
        "Use decorator.decorator() to preserve function signatures",
        "Handle edge cases and error conditions in decorators",
        "Make decorators configurable with parameters when useful",
        "Consider performance impact of decorators in hot code paths",
        "Use functools.wraps() for traditional decorators as fallback",
        "Document decorator behavior and side effects clearly",
        "Test decorated functions with various inputs and edge cases"
    ]
    
    for practice in best_practices:
        print(f"  • {practice}")
    
    # Integration examples
    print("✓ Integration examples:")
    
    integrations = [
        "Flask/Django: Request validation, authentication, rate limiting",
        "FastAPI: Dependency injection, request/response processing",
        "Click: Command-line option validation and processing",
        "SQLAlchemy: Transaction management, query optimization",
        "Celery: Task retry logic, result caching",
        "Testing: Mock setup/teardown, test data generation"
    ]
    
    for integration in integrations:
        print(f"  • {integration}")
    
    # Performance considerations
    print("✓ Performance tips:")
    
    performance_tips = [
        "Minimize decorator overhead for frequently called functions",
        "Use class-based decorators for stateful behavior",
        "Consider decorator stacking order for optimal performance",
        "Profile decorated code to identify bottlenecks",
        "Use conditional decorators based on environment/config"
    ]
    
    for tip in performance_tips:
        print(f"  • {tip}")
    
    print("✓ Real-world applications and best practices completed successfully")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("  Note: decorator package is essential for production Python applications")
except Exception as e:
    print(f"❌ Runtime error: {e}")
    raise AssertionError(
        f"decorator example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 5 finished")